In [1]:
import pandas as pd

# Paths to raw datasets
stock_path = "../../data/raw/stock_prices.csv"
index_path = "../../data/raw/sp500_index.csv"
companies_path = "../../data/raw/sp500_companies.csv"


def validate_dataset(name, path):
    
    print("\n" + "="*70)
    print(f"DATASET VALIDATION REPORT: {name}")
    print("="*70)

    df = pd.read_csv(path)

    # Shape
    print("\nDataset Shape (Rows, Columns):")
    print(df.shape)

    # Column Names
    print("\nColumn Names:")
    print(df.columns.tolist())

    # Data Types
    print("\nData Types:")
    print(df.dtypes)

    # Missing Values
    print("\nMissing Values:")
    print(df.isnull().sum())

    # Duplicate Rows
    duplicates = df.duplicated().sum()
    print("\nDuplicate Rows:", duplicates)

    # Sample Data
    print("\nSample Records:")
    print(df.head())

    # Date column check
    possible_date_cols = [col for col in df.columns if "date" in col.lower()]
    
    if possible_date_cols:
        print("\nDate Columns Detected:", possible_date_cols)

    # Potential Join Keys
    potential_keys = []
    for col in df.columns:
        if "symbol" in col.lower() or "ticker" in col.lower():
            potential_keys.append(col)

    if potential_keys:
        print("\nPotential Join Keys:", potential_keys)

    print("\nValidation complete.")


# Validate each dataset
validate_dataset("Stock Prices", stock_path)
validate_dataset("S&P 500 Index", index_path)
validate_dataset("S&P 500 Companies", companies_path)


DATASET VALIDATION REPORT: Stock Prices

Dataset Shape (Rows, Columns):
(12581, 27)

Column Names:
['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker', 'Close.1', 'High.1', 'Low.1', 'Open.1', 'Volume.1', 'Close.2', 'High.2', 'Low.2', 'Open.2', 'Volume.2', 'Close.3', 'High.3', 'Low.3', 'Open.3', 'Volume.3', 'Close.4', 'High.4', 'Low.4', 'Open.4', 'Volume.4']

Data Types:
Date        object
Close       object
High        object
Low         object
Open        object
Volume      object
Ticker      object
Close.1     object
High.1      object
Low.1       object
Open.1      object
Volume.1    object
Close.2     object
High.2      object
Low.2       object
Open.2      object
Volume.2    object
Close.3     object
High.3      object
Low.3       object
Open.3      object
Volume.3    object
Close.4     object
High.4      object
Low.4       object
Open.4      object
Volume.4    object
dtype: object

Missing Values:
Date            1
Close       10064
High        10064
Low         10064
Op

# Data Validation Findings and Observations

This section summarizes the results of the validation performed on the three raw datasets collected for the ETL pipeline. The goal of this validation step was to understand the structure, quality, and compatibility of the datasets before beginning the cleaning and integration stages.

---

# 1. Stock Prices Dataset

### Dataset Structure
- Rows: 12,581  
- Columns: 27  

The dataset contains historical stock prices for multiple companies. The companies included are:

- AAPL (Apple)
- MSFT (Microsoft)
- AMZN (Amazon)
- NVDA (NVIDIA)
- TSLA (Tesla)

Each company has its own set of columns for:

- Close
- High
- Low
- Open
- Volume

Because of this structure, the dataset currently exists in a **wide format**, where each ticker's values are stored in separate column groups.

### Problems Identified

**1. Multi-Ticker Wide Format**

Columns are repeated for each company:

Close, High, Low, Open, Volume  
Close.1, High.1, Low.1, Open.1, Volume.1  
Close.2, High.2, Low.2, Open.2, Volume.2  

This format is not suitable for analytics or SQL joins.

The dataset must be transformed into a **long format**, where each row represents:



Example of desired structure:

| Date | Ticker | Open | High | Low | Close | Volume |
|-----|-----|-----|-----|-----|-----|-----|

---

**2. Incorrect First Row**

The first row contains ticker labels instead of actual data values.

Example:



This row must be removed during cleaning.

---

**3. Incorrect Data Types**

All price columns are currently stored as:



They should be converted to:

- Float (price values)
- Integer (volume)

---

**4. Large Number of Missing Values**

Many columns contain missing values because each ticker is stored in separate column groups.

These missing values will naturally disappear once the dataset is reshaped into long format.

---

# 2. S&P 500 Index Dataset

### Dataset Structure

- Rows: 2,517
- Columns: 6

Columns include:

- Date
- Close
- High
- Low
- Open
- Volume

### Problems Identified

**1. Incorrect First Row**

The first row contains ticker labels:



This row must be removed during cleaning.

---

**2. Data Type Issues**

All columns are currently stored as:



They must be converted to:

- Date → datetime
- Prices → float
- Volume → integer

---

# 3. S&P 500 Companies Dataset

### Dataset Structure

- Rows: 503
- Columns: 8

Columns include:

- Symbol
- Security
- GICS Sector
- GICS Sub-Industry
- Headquarters Location
- Date Added
- CIK
- Founded

### Observations

This dataset is already well structured and contains **no missing values or duplicate rows**.

Only minor adjustments are needed, such as renaming columns for consistency.

Recommended column names:

| Current Name | Clean Name |
|------|------|
Symbol | Ticker |
Security | Company |
GICS Sector | Sector |
GICS Sub-Industry | Industry |

---

# Dataset Granularity

Understanding dataset granularity is essential for integration.

| Dataset | Granularity |
|------|------|
Stock Prices | Date + Ticker |
S&P 500 Index | Date |
S&P 500 Companies | Ticker |

This confirms that the datasets are **compatible for merging**.

---

# Integration Strategy

The datasets will be integrated using the following keys:

### 1. Join Stock Prices with Company Metadata


This join will allow us to add:

- Company name
- Sector
- Industry

to each stock price record.

---

### 2. Join Stock Prices with Market Index


This join allows us to compare individual stock performance with the overall market benchmark.

---

# Final Analytical Dataset

After cleaning and integration, the final dataset will have the following structure:

| Date | Ticker | Company | Sector | Open | High | Low | Close | Volume | Market_Close |
|------|------|------|------|------|------|------|------|------|------|

This dataset will support:

- sector analysis
- stock vs market comparison
- time-series analysis
- dashboard creation in Power BI

---

# Next Steps

Based on the validation results, the next phase of the ETL pipeline will include:

### 1. Data Cleaning Scripts
Separate cleaning scripts will be created for each dataset:


These scripts will handle:

- removing invalid header rows
- converting data types
- renaming columns
- reshaping the stock dataset from wide to long format

---

### 2. Data Integration
After cleaning, SQL will be used to join the datasets and create a final integrated dataset ready for analysis.

---

# Conclusion

The validation step confirms that the three datasets are compatible and suitable for building an integrated financial analysis dataset.

Although structural issues exist in the raw data, they can be resolved through systematic cleaning and transformation. Once cleaned, the datasets will support a robust ETL pipeline and enable advanced financial analytics.
